# TML — Gallica Year Reconstruction Universal
Always-on ALTO lane + optional GPU compute. The ALTO downloader works even when Colab does not grant a GPU; when CUDA is available the same session also runs the 4-worker RapidOCR / layout compute lane.


In [ ]:
YEAR=1906
POLL_SECONDS=30
ALTO_DELAY=12.0
ALTO_JITTER_MAX=0.25
RAPID_WORKERS=4
RAPID_DOWNLOADERS=4
LAYOUT_WORKERS=1
LAYOUT_DOWNLOADERS=4
VPS_HOST='vibrant-lovelace.82-165-11-122.plesk.page'
VPS_USER='andre'
VPS_PORT=2222
import subprocess
HAS_GPU=False; GPU_NAME='NONE'; GPU_MEM='0'
try:
    gpu_line=subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader,nounits'],text=True,stderr=subprocess.DEVNULL).strip().splitlines()[0]
    GPU_NAME,GPU_MEM=[x.strip() for x in gpu_line.rsplit(',',1)]; HAS_GPU=True
except Exception:
    pass
BASE=f'/home/andre/GallicaJobs/gallica-{YEAR}-all-tennis/GALlica_{YEAR}_ALL_TENNIS'
print('TML_SESSION_CONFIG','YEAR',YEAR,'GPU',GPU_NAME,'GPU_AVAILABLE',HAS_GPU,'ALTO_ALWAYS_ON',True,'ALTO_DELAY',ALTO_DELAY,flush=True)


In [ ]:
import os,shutil,subprocess,sys
REPO='/content/Tennis-OCR-Pipeline'
RAPID_ENV='/content/tml-rapid-env'
LAYOUT_ENV='/content/tml-layout-env'
RAPID_PY=f'{RAPID_ENV}/bin/python'
LAYOUT_PY=f'{LAYOUT_ENV}/bin/python'
if os.path.isdir(REPO): subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
else: subprocess.run(['git','clone','-q','https://github.com/Tennismylife/Tennis-OCR-Pipeline.git',REPO],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','paramiko>=3.5,<4'],check=True)
if HAS_GPU:
    subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
    UV=shutil.which('uv'); assert UV
    if not os.path.exists(RAPID_PY): subprocess.run([UV,'venv','--seed',RAPID_ENV],check=True)
    subprocess.run([RAPID_PY,'-m','pip','install','-q','-r',f'{REPO}/colab/requirements.txt'],check=True)
    subprocess.run([RAPID_PY,'-c',"import onnxruntime as o; p=o.get_available_providers(); print('RAPID_GPU_ENV_READY',o.__version__,p); assert 'CUDAExecutionProvider' in p"],check=True)
    subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],check=True)
else:
    print('NO_GPU: compute lane disabled; ALTO downloader will still run continuously',flush=True)


In [ ]:
from google.colab import files
import os
uploaded=files.upload()
if not uploaded: raise RuntimeError('Upload the dedicated Colab SFTP private key')
name,data=next(iter(uploaded.items()))
if name.endswith('.pub'): raise RuntimeError('Upload the private key, not .pub')
KEY_FILE='/content/tml_colab_key'
open(KEY_FILE,'wb').write(data)
os.chmod(KEY_FILE,0o600)
print('KEY_READY',name,flush=True)


In [ ]:
import importlib.util,subprocess,sys
subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
commit=subprocess.check_output(['git','-C',REPO,'rev-parse','--short','HEAD'],text=True).strip()
print('CODE',commit,flush=True)
if HAS_GPU:
    path=f'{REPO}/colab/compute_supervisor.py'
    spec=importlib.util.spec_from_file_location('tml_compute_supervisor',path)
    mod=importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
    argv=['compute_supervisor.py','--year',str(YEAR),'--vps-host',VPS_HOST,'--vps-user',VPS_USER,'--vps-port',str(VPS_PORT),'--vps-key-file',KEY_FILE,'--repo',REPO,'--rapid-python',RAPID_PY,'--layout-python',LAYOUT_PY,'--compute-device','cuda','--rapid-workers',str(RAPID_WORKERS),'--rapid-downloaders',str(RAPID_DOWNLOADERS),'--layout-workers',str(LAYOUT_WORKERS),'--layout-downloaders',str(LAYOUT_DOWNLOADERS),'--poll',str(POLL_SECONDS),'--alto-delay',str(ALTO_DELAY),'--alto-jitter-max',str(ALTO_JITTER_MAX)]
    print('STARTING ALTO_ALWAYS_ON + GPU_COMPUTE',flush=True)
    old_argv=sys.argv[:]; sys.argv=argv
    try: mod.main()
    finally: sys.argv=old_argv
else:
    cmd=[sys.executable,'-u',f'{REPO}/colab/alto_watch_filekey.py','--year',str(YEAR),'--vps-host',VPS_HOST,'--vps-user',VPS_USER,'--vps-port',str(VPS_PORT),'--vps-key-file',KEY_FILE,'--delay',str(ALTO_DELAY),'--jitter-min','0','--jitter-max',str(ALTO_JITTER_MAX),'--poll',str(max(5,POLL_SECONDS))]
    print('STARTING ALTO_ALWAYS_ON ONLY (NO GPU REQUIRED)',flush=True)
    raise SystemExit(subprocess.call(cmd))
